# Tennis — Palier 1 : détection + suivi (tracking)

Objectif : à partir d'une vidéo de tennis, **détecter** les joueurs et la balle sur chaque frame, les **suivre** dans le temps (ID stable), tracer la **trajectoire de la balle**, et produire une **vidéo annotée**.

On n'entraîne **rien** ici : on utilise un YOLO **pré-entraîné sur COCO**, qui connaît déjà les classes `person` et `sports ball`. C'est la base sur laquelle s'appuieront les paliers 2 (gestes) et 3 (comptage).

> ⚠️ Kernel : `.venv` (où ultralytics/torch sont installés).

## 1. Configuration et découpe d'un rallye

Le dataset **TenniSet** (dans `data/videos/`) contient des matchs complets (~80 min). Inutile de tout traiter : le fichier `data/annotations/points.txt` donne les **bornes de frames de chaque point**. On extrait ici un seul rallye (~10 s) de `V006` dans `data/tennis.mp4` — reproductible, sans coupe au jugé.

In [ ]:
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

VIDEO_PATH = Path('data/tennis.mp4')      # clip de rallye (généré ci-dessous)
OUTPUT_PATH = Path('data/tennis_annote.mp4')

# Classes COCO qui nous intéressent
PERSON, SPORTS_BALL = 0, 32

# --- Découpe reproductible d'un rallye depuis TenniSet ---
# points.txt : id | video | frame_debut | frame_fin | numero. Ici le 1er point de V006.
SOURCE_VIDEO = Path('data/videos/V006.mp4')
RALLY_START, RALLY_END = 25740, 25984

def cut_rally():
    if VIDEO_PATH.exists():
        print('Clip déjà présent :', VIDEO_PATH)
        return
    cap = cv2.VideoCapture(str(SOURCE_VIDEO))
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    wri = cv2.VideoWriter(str(VIDEO_PATH), cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    cap.set(cv2.CAP_PROP_POS_FRAMES, RALLY_START)
    for _ in range(RALLY_START, RALLY_END + 1):
        ok, f = cap.read()
        if not ok:
            break
        wri.write(f)
    cap.release(); wri.release()
    print(f'Clip créé : {VIDEO_PATH} ({RALLY_END - RALLY_START + 1} frames)')

cut_rally()
assert VIDEO_PATH.exists(), f'Vidéo introuvable : {VIDEO_PATH}'
print('Vidéo :', VIDEO_PATH)

## 2. Charger le modèle et tester sur une frame

`yolo11n.pt` (version *nano*, la plus légère) se télécharge automatiquement au premier appel. On vérifie d'abord ce qu'il détecte sur une seule image.

In [ ]:
model = YOLO('yolo11n.pt')

# Lire la première frame
cap = cv2.VideoCapture(str(VIDEO_PATH))
ok, frame = cap.read()
cap.release()
assert ok, 'Impossible de lire la vidéo.'

res = model(frame, classes=[PERSON, SPORTS_BALL], verbose=False)[0]
annotated = res.plot()  # BGR
plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title(f'{len(res.boxes)} objets détectés sur la 1re frame')
plt.axis('off'); plt.show()

## 3. Détection + suivi sur toute la vidéo

`model.track(..., persist=True)` relie les détections d'une frame à l'autre et attribue un **ID** stable à chaque objet (via ByteTrack). On :
- écrit une **vidéo annotée** (boîtes + IDs),
- mémorise la **position du centre** de chaque objet à chaque frame (pour les trajectoires).

In [ ]:
cap = cv2.VideoCapture(str(VIDEO_PATH))
fps = cap.get(cv2.CAP_PROP_FPS) or 30
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

writer = cv2.VideoWriter(str(OUTPUT_PATH), cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

# trajectoires : {track_id: [(x, y, frame_idx), ...]}
tracks = defaultdict(list)
ball_positions = []  # (x, y, frame_idx) pour la balle

results = model.track(
    source=str(VIDEO_PATH),
    stream=True,            # générateur frame par frame (pas tout en RAM)
    persist=True,
    classes=[PERSON, SPORTS_BALL],
    tracker='bytetrack.yaml',
    conf=0.25,
    verbose=False,
)

for frame_idx, r in enumerate(results):
    writer.write(r.plot())  # frame annotée
    if r.boxes.id is None:
        continue
    ids = r.boxes.id.int().cpu().tolist()
    clss = r.boxes.cls.int().cpu().tolist()
    xywh = r.boxes.xywh.cpu().numpy()   # centre (cx, cy) + taille
    for tid, cl, (cx, cy, _, _) in zip(ids, clss, xywh):
        tracks[tid].append((float(cx), float(cy), frame_idx))
        if cl == SPORTS_BALL:
            ball_positions.append((float(cx), float(cy), frame_idx))

writer.release()
print(f'{frame_idx + 1} frames traitées.')
print(f'{len(tracks)} identités suivies, {len(ball_positions)} détections de balle.')
print('Vidéo annotée :', OUTPUT_PATH)

## 4. Tracé des trajectoires

In [ ]:
# On superpose les trajectoires sur la 1re frame
cap = cv2.VideoCapture(str(VIDEO_PATH))
ok, bg = cap.read(); cap.release()
bg = cv2.cvtColor(bg, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 7))
plt.imshow(bg)

# Trajectoires des personnes (longues) en lignes, balle en points
for tid, pts in tracks.items():
    if len(pts) < 10:            # on ignore les pistes trop courtes (bruit)
        continue
    xs = [p[0] for p in pts]; ys = [p[1] for p in pts]
    plt.plot(xs, ys, linewidth=1.5, alpha=0.7)

if ball_positions:
    bx = [p[0] for p in ball_positions]; by = [p[1] for p in ball_positions]
    plt.scatter(bx, by, c='yellow', s=12, edgecolors='black', linewidths=0.3, label='balle')
    plt.legend()

plt.title('Trajectoires (joueurs = lignes, balle = points jaunes)')
plt.axis('off'); plt.show()

## 5. Petites statistiques

In [ ]:
# Durée de présence de chaque identité (en frames), triée
durees = sorted(((tid, len(pts)) for tid, pts in tracks.items()), key=lambda x: -x[1])
print('Top identités par durée de suivi (frames) :')
for tid, n in durees[:8]:
    print(f'  ID {tid:>3} : {n} frames  (~{n / fps:.1f} s)')

# Les 2 pistes les plus longues = probablement les 2 joueurs
print('\n→ Les identités les plus longues correspondent en général aux 2 joueurs.')

## 6. Ce qu'on observe et les limites

- **Les joueurs** se suivent bien (gros objets, mouvement régulier).
- **La balle** est le point dur : petite, rapide, floue → le YOLO COCO générique la rate souvent (détections intermittentes dans `ball_positions`). C'est normal et attendu.

**Pistes pour fiabiliser la balle** (si tu veux pousser le palier 1) :
- Un modèle **spécialisé balle de tennis** (dataset Roboflow « tennis ball » à fine-tuner sur YOLO) au lieu du COCO générique.
- **TrackNet**, une archi conçue exprès pour suivre la balle de tennis malgré le flou.
- **Interpoler/lisser** la trajectoire entre les détections manquantes (filtre de Kalman).

Une fois le suivi en place, le **palier 2** classera les gestes (service / coup droit / revers) à partir des séquences de poses des joueurs.